# 07. Review Decision Test

06번 refinement 후보를 운영자 검토용 decision table에 저장합니다.

In [ ]:
import sys
import importlib

from pyspark.sql import functions as F

PROJECT_ROOT = "/Workspace/Users/jungryo.lee@lge.com/prj_TV_voc"
SRC_ROOT = f"{PROJECT_ROOT}/src"

if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)

import common.config_loader as config_loader
import taxonomy.topic_pool_refiner as topic_pool_refiner
import taxonomy.review_decision_writer as review_decision_writer

importlib.reload(config_loader)
importlib.reload(topic_pool_refiner)
importlib.reload(review_decision_writer)

from common.config_loader import load_config, get_output_table
from taxonomy.topic_pool_refiner import refine_topic_pool_candidates
from taxonomy.review_decision_writer import build_review_decision_df, save_review_decisions

config = load_config(f"{PROJECT_ROOT}/config/settings.yaml")
print("config loaded")

In [ ]:
CLASSIFICATION_INPUT_TABLE_KEY = "classification_full"

MODEL_KEY = "gpt_55"
MODEL_VERSION_VALUE = config["llm"]["models"][MODEL_KEY]["model_version"]
TARGET_CATE_1_DEPTH = "07. 스마트 사용성"
TARGET_CATE_2_DEPTH = "07-01. 채널/컨텐츠 APP"
TARGET_SC_MEASUREMENT = 1

refinement_result = refine_topic_pool_candidates(
    spark=spark,
    config=config,
    input_table_key=CLASSIFICATION_INPUT_TABLE_KEY,
    cate_1_depth=TARGET_CATE_1_DEPTH,
    cate_2_depth=TARGET_CATE_2_DEPTH,
    sc_measurement=TARGET_SC_MEASUREMENT,
    model_version=MODEL_VERSION_VALUE,
    prompt_version=config["version"]["prompt_version"],
    taxonomy_version=config["version"]["taxonomy_version"],
)

print({
    "input_table_key": refinement_result["input_table_key"],
    "existing_reassignment_cnt": refinement_result["existing_topic_reassignment_df"].count(),
    "new_topic_candidate_cnt": refinement_result["new_topic_candidate_df"].count(),
})

In [ ]:
decision_df = build_review_decision_df(
    spark=spark,
    config=config,
    existing_topic_reassignment_df=refinement_result["existing_topic_reassignment_df"],
    new_topic_candidate_df=refinement_result["new_topic_candidate_df"],
    source_table_key=CLASSIFICATION_INPUT_TABLE_KEY,
    decision_status="pending",
    created_by="codex",
)

print({"decision_cnt": decision_df.count()})

display(
    decision_df.select(
        "decision_id",
        "candidate_type",
        "cate_2_depth",
        "sc_measurement",
        "sample_memo",
        "suggested_action",
        "suggested_topic",
        "candidate_cnt",
        "candidate_distinct_memo_id_cnt",
        "candidate_ratio",
        "decision_status",
    )
    .orderBy(F.col("candidate_type"), F.col("candidate_cnt").desc_nulls_last())
)

In [ ]:
saved_table = save_review_decisions(
    spark=spark,
    config=config,
    decision_df=decision_df,
    write_mode="replace_decisions",
)

print("saved_table =", saved_table)

In [ ]:
review_decision_table = get_output_table(config, "review_decision")

display(
    spark.table(review_decision_table)
    .where(F.col("cate_1_depth") == TARGET_CATE_1_DEPTH)
    .where(F.col("cate_2_depth") == TARGET_CATE_2_DEPTH)
    .where(F.col("sc_measurement") == TARGET_SC_MEASUREMENT)
    .where(F.col("model_version") == MODEL_VERSION_VALUE)
    .where(F.col("prompt_version") == config["version"]["prompt_version"])
    .where(F.col("taxonomy_version") == config["version"]["taxonomy_version"])
    .orderBy(F.col("created_at").desc(), F.col("candidate_type").asc())
)